# Environment Setup & Model Loading

This notebook does the *environment + model loading* half of O2 and ends with a smoke test that proves the inference path works. 

**Colab GPU:** Prefer L4 or A100


## 1. Check the GPU 


In [7]:
import torch
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > GPU.'
p = torch.cuda.get_device_properties(0)
print(f'GPU: {p.name}  sm_{p.major}{p.minor}  {p.total_memory/1024**3:.1f} GB')
print('FlashAttention-2 / native bf16 available:' , (p.major, p.minor) >= (8, 0))

GPU: NVIDIA A100-SXM4-80GB  sm_80  79.3 GB
FlashAttention-2 / native bf16 available: True


## 2. Mount Google Drive

Caching the HF weights in Drive.


In [8]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['HF_HOME'] = '/content/drive/MyDrive/openvla_cache/hf'
os.makedirs(os.environ['HF_HOME'], exist_ok=True)
print('HF cache ->', os.environ['HF_HOME'])

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
HF cache -> /content/drive/MyDrive/openvla_cache/hf


## 3. Install Pinned Dependencies

> Do **not** reinstall torch, Colab's build is matched to its CUDA driver.


In [9]:
 !pip install -q transformers==4.40.1 tokenizers==0.19.1 timm==0.9.10 \
    huggingface_hub==0.23.4 accelerate==0.30.1 'bitsandbytes>=0.45.0'

**After this install, restart the runtime once** because Colab pre-imports a newer transformers. Then re-run cells 1–2 and skip this install cell.


## 4. Load OpenVLA-7B

4-bit NF4 by default, `eager` attention for T4 portability, compute dtype chosen from the GPU.

In [10]:
import torch
from PIL import Image
import numpy as np
from transformers import AutoModelForVision2Seq, AutoProcessor, BitsAndBytesConfig

cap = torch.cuda.get_device_properties(0)
compute_dtype = torch.bfloat16 if (cap.major, cap.minor) >= (8, 0) else torch.float16
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_use_double_quant=True,
                         bnb_4bit_compute_dtype=compute_dtype)
processor = AutoProcessor.from_pretrained('openvla/openvla-7b', trust_remote_code=True)
vla = AutoModelForVision2Seq.from_pretrained(
    'openvla/openvla-7b', quantization_config=bnb, attn_implementation='eager',
    torch_dtype=compute_dtype, low_cpu_mem_usage=True, trust_remote_code=True,
    device_map={'': 0})
vla.eval()

def predict_action(processor, vla, image, instruction, compute_dtype,
                   unnorm_key='bridge_orig', do_sample=False):
    prompt = f'In: What action should the robot take to {instruction}?\nOut:'
    inputs = processor(prompt, image.convert('RGB')).to('cuda:0', dtype=compute_dtype)
    inputs.pop('attention_mask', None)   # drop stale mask; predict_action appends a token and desyncs it
    a = vla.predict_action(**inputs, unnorm_key=unnorm_key, do_sample=do_sample)
    return np.asarray(a, dtype=np.float32)

print('Loaded. Allocated:', round(torch.cuda.memory_allocated(0)/1024**3, 2), 'GB')

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Loaded. Allocated: 8.17 GB


## 5. Smoke Test

A synthetic frame is fine here; we're only checking that the pipeline produces a
well-formed 7-DoF action `[dx, dy, dz, droll, dpitch, dyaw, gripper]`. Real
BridgeData V2 frames come in the next notebook.


In [11]:
dummy = Image.fromarray((np.random.default_rng(0).random((224,224,3))*255).astype(np.uint8))
action = predict_action(processor, vla, dummy, 'pick up the object on the left', compute_dtype)
print('Action shape :', action.shape)
print('Action vector:', np.round(action, 4))
assert action.shape == (7,), 'Expected a 7-DoF vector'

Action shape : (7,)
Action vector: [-0.0029  0.0158 -0.0135 -0.0084 -0.0584 -0.0017  0.9961]


## 6. Sanity Check

Same image, two instructions differing only in the spatial term. With a random
image, a clean sign flip should not be expected. This cell just confirms the
probe mechanics (two calls, compare `dx`) work before real scenes are wired in.


In [12]:
left  = predict_action(processor, vla, dummy, 'move to the cup on the left',  compute_dtype)
right = predict_action(processor, vla, dummy, 'move to the cup on the right', compute_dtype)
print('dx(left) = %.4f   dx(right) = %.4f   diff = %.4f' % (left[0], right[0], left[0]-right[0]))

dx(left) = -0.0078   dx(right) = 0.0031   diff = -0.0110
